# Task 5 — Final Report, Automation & Presentation
**ApexPlanet Data Analytics Internship**  
**Dataset:** E-commerce Sales (Cleaned)  
**Timeline:** 4 Days (Day 27–30)

---

## 📋 What We Cover
| Days | Topic |
|------|-------|
| Day 27 | Executive Summary PDF Report |
| Day 28–29 | Automate Pipeline (Python Script + Excel Export) |
| Day 30 | GitHub Cleanup & Final Submission |

---

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
from fpdf import FPDF
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, LineChart, Reference

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load Cleaned Dataset

In [ ]:
df = pd.read_csv('../data/data_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
df.head(3)

---
## 📊 Step 3: Compute All KPIs

In [ ]:
# Core KPIs
total_revenue   = df['TotalPrice'].sum()
total_orders    = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
total_products  = df['Description'].nunique()
avg_order_value = total_revenue / total_orders
top_country     = df.groupby('Country')['TotalPrice'].sum().idxmax()

# Monthly data
monthly = (
    df.groupby(['Year', 'Month'])
    .agg(Revenue=('TotalPrice', 'sum'),
         Orders=('InvoiceNo', 'nunique'),
         Customers=('CustomerID', 'nunique'))
    .reset_index()
)
monthly['Period'] = pd.to_datetime(
    monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)
)
monthly.sort_values('Period', inplace=True)
peak_month = monthly.loc[monthly['Revenue'].idxmax()]

# Top items
top_countries = (
    df.groupby('Country')['TotalPrice'].sum()
    .sort_values(ascending=False).head(10).reset_index()
)
top_products = (
    df.groupby('Description')['TotalPrice'].sum()
    .sort_values(ascending=False).head(10).reset_index()
)
top_customers = (
    df.groupby('CustomerID')['TotalPrice'].sum()
    .sort_values(ascending=False).head(10).reset_index()
)

print('📊 KPI Summary')
print(f'  Total Revenue   : £{total_revenue:,.2f}')
print(f'  Total Orders    : {total_orders:,}')
print(f'  Total Customers : {total_customers:,}')
print(f'  Unique Products : {total_products:,}')
print(f'  Avg Order Value : £{avg_order_value:,.2f}')
print(f'  Top Country     : {top_country}')
print(f'  Peak Month      : {peak_month["Period"].strftime("%B %Y")}')

---
# 📅 DAY 27: EXECUTIVE SUMMARY PDF REPORT

### Step 4 — Generate Supporting Chart for Report

In [ ]:
# Monthly revenue chart — used inside the PDF
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['Period'], monthly['Revenue'],
        color='#1565C0', linewidth=2.5, marker='o', markersize=5)
ax.fill_between(monthly['Period'], monthly['Revenue'],
                alpha=0.12, color='#1565C0')
ax.set_title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/report_monthly_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Chart saved for PDF report')

### Step 5 — Build the PDF Report

In [ ]:
class EcommercePDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 13)
        self.set_text_color(21, 101, 192)
        self.cell(0, 9, 'ApexPlanet Data Analytics Internship',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.set_font('Helvetica', '', 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 6, 'E-Commerce Sales Analysis - Final Executive Report',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.ln(2)
        self.set_draw_color(21, 101, 192)
        self.set_line_width(0.5)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-13)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10,
                  f'Page {self.page_no()} | ApexPlanet Software Pvt. Ltd. | www.apexplanet.in',
                  align='C')

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_text_color(21, 101, 192)
        self.cell(0, 9, title, new_x='LMARGIN', new_y='NEXT')
        self.set_draw_color(200, 220, 255)
        self.line(15, self.get_y(), 195, self.get_y())
        self.ln(3)

pdf = EcommercePDF()
pdf.set_margins(15, 15, 15)
pdf.add_page()

# Section 1: Executive Summary
pdf.section_title('1. Executive Summary')
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(50, 50, 50)
summary = (
    f'This report presents a comprehensive analysis of E-commerce Sales data '
    f'conducted as part of the 30-Day Data Analytics Internship at ApexPlanet '
    f'Software Pvt. Ltd. The dataset comprised {df.shape[0]:,} transactions from '
    f'a UK-based online retailer. Through data cleaning, SQL extraction, '
    f'visualization, statistical testing, customer segmentation, and predictive '
    f'modeling, actionable business insights were uncovered to support data-driven '
    f'decision making across marketing, operations, and customer success.'
)
pdf.multi_cell(0, 6, summary)
pdf.ln(5)

# Section 2: KPI Cards
pdf.section_title('2. Key Performance Indicators')
kpis = [
    ('Total Revenue (GBP)',  f'GBP {total_revenue:,.2f}'),
    ('Total Orders',         f'{total_orders:,}'),
    ('Unique Customers',     f'{total_customers:,}'),
    ('Unique Products',      f'{total_products:,}'),
    ('Avg Order Value',      f'GBP {avg_order_value:,.2f}'),
    ('Top Country',          top_country),
    ('Peak Sales Month',     peak_month['Period'].strftime('%B %Y')),
    ('Total Rows Analysed',  f'{df.shape[0]:,}'),
]
pdf.set_font('Helvetica', '', 10)
for label, value in kpis:
    pdf.set_fill_color(232, 240, 254)
    pdf.set_text_color(21, 101, 192)
    pdf.cell(80, 8, f'  {label}', border=0, fill=True)
    pdf.set_text_color(40, 40, 40)
    pdf.cell(100, 8, f'  {value}', border=0, new_x='LMARGIN', new_y='NEXT')
    pdf.ln(1)
pdf.ln(5)

# Section 3: Monthly Chart
pdf.section_title('3. Monthly Revenue Trend')
if os.path.exists('../reports/report_monthly_chart.png'):
    pdf.image('../reports/report_monthly_chart.png', x=15, w=175)
pdf.ln(4)

# Page 2
pdf.add_page()

# Section 4: Top 5 Insights
pdf.section_title('4. Top 5 Key Insights')
insights = [
    ('Seasonal Revenue Peak',
     f'{peak_month["Period"].strftime("%B %Y")} was the peak revenue month. '
     f'Q4 consistently generates the highest sales - marketing investment '
     f'should be concentrated in October-November.'),
    ('Geographic Concentration',
     f'{top_country} dominates revenue contribution. International markets '
     f'represent an untapped growth opportunity with targeted expansion campaigns.'),
    ('Customer Segmentation (K-Means, K=4)',
     '4 distinct customer tiers identified: Champions, Loyal, At-Risk, and '
     'Occasional - each requiring a tailored engagement and retention strategy.'),
    ('Statistical Spending Differences',
     'T-Test confirmed UK and Non-UK customers have statistically different '
     'spending behaviors (p < 0.05), validating the need for region-specific pricing.'),
    ('Predictive Model Performance',
     'Logistic Regression achieved strong accuracy predicting high-value orders. '
     'UnitPrice was identified as the strongest single predictor of order value.'),
]
pdf.set_font('Helvetica', '', 10)
for i, (title, detail) in enumerate(insights, 1):
    pdf.set_font('Helvetica', 'B', 10)
    pdf.set_text_color(21, 101, 192)
    pdf.cell(0, 7, f'  {i}. {title}', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(60, 60, 60)
    pdf.multi_cell(0, 5, f'     {detail}')
    pdf.ln(2)
pdf.ln(3)

# Section 5: Business Recommendations
pdf.section_title('5. Business Recommendations')
recommendations = [
    ('Invest in Q4 Marketing',
     'Allocate 40%+ of the annual marketing budget to Q4 campaigns. Flash sales, '
     'bundle deals, and email sequences during Oct-Nov maximise natural revenue surge.'),
    ('Launch a Customer Loyalty Program',
     'Champions and Loyal customers drive disproportionate revenue. A tiered loyalty '
     'program with exclusive discounts and early product access increases lifetime value.'),
    ('Win-Back At-Risk Customers',
     'Deploy targeted win-back campaigns for At-Risk segments using time-limited '
     'discount vouchers and personalised product recommendations based on past purchases.'),
]
for i, (title, detail) in enumerate(recommendations, 1):
    pdf.set_font('Helvetica', 'B', 10)
    pdf.set_text_color(21, 101, 192)
    pdf.cell(0, 7, f'  {i}. {title}', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(60, 60, 60)
    pdf.multi_cell(0, 5, f'     {detail}')
    pdf.ln(3)
pdf.ln(2)

# Section 6: Tasks Completed
pdf.section_title('6. Internship Tasks Completed')
tasks = [
    ('Task 1', 'Foundational Setup & Exploratory Data Analysis'),
    ('Task 2', 'SQL for Data Extraction'),
    ('Task 3', 'Data Visualization & Dashboarding'),
    ('Task 4', 'Advanced Analytics (Basic)'),
    ('Task 5', 'Final Report, Automation & Presentation'),
]
pdf.set_font('Helvetica', '', 9)
for task, desc in tasks:
    pdf.set_text_color(21, 101, 192)
    pdf.cell(22, 7, f'  {task}')
    pdf.set_text_color(50, 50, 50)
    pdf.cell(140, 7, desc)
    pdf.set_text_color(0, 140, 0)
    pdf.cell(18, 7, 'Complete', new_x='LMARGIN', new_y='NEXT')

pdf.ln(5)
pdf.set_font('Helvetica', 'I', 8)
pdf.set_text_color(130, 130, 130)
pdf.multi_cell(0, 5,
    'Analysis performed using Python (Pandas, Matplotlib, Seaborn, Plotly, '
    'Scikit-Learn, SciPy, SQLite) on the E-commerce Sales dataset (Kaggle). '
    'ApexPlanet Software Pvt. Ltd. - www.apexplanet.in'
)

# Save PDF
pdf.output('../reports/final_executive_report.pdf')
print('PDF saved -> reports/final_executive_report.pdf')


---
# 📅 DAY 28–29: AUTOMATE PIPELINE

### Step 6 — Automated Python Pipeline Script

In [ ]:
pipeline_script = '''
"""
ApexPlanet Data Analytics Internship — Task 5
Automated Data Pipeline Script
Runs: Load → Clean → Analyse → Export to Excel → Save Charts
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

warnings.filterwarnings("ignore")

print("=" * 55)
print("   APEXPLANET — AUTOMATED DATA PIPELINE")
print(f"   Run Time: {datetime.now().strftime(\\%Y-%m-%d %H:%M:%S\\)}")
print("=" * 55)

# ── STEP 1: LOAD RAW DATA ──
print("\\n[1/5] Loading raw data...")
df = pd.read_csv("../data/data.csv", encoding="ISO-8859-1")
print(f"      Loaded: {len(df):,} rows")

# ── STEP 2: CLEAN DATA ──
print("[2/5] Cleaning data...")
df.drop_duplicates(inplace=True)
df.dropna(subset=["CustomerID"], inplace=True)
df["Description"].fillna("Unknown", inplace=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"]  = df["CustomerID"].astype(int).astype(str)
df["Year"]  = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"]   = df["InvoiceDate"].dt.day
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
for col in ["Quantity", "UnitPrice"]:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df.to_csv("../data/data_cleaned_auto.csv", index=False)
print(f"      Clean rows: {len(df):,} | Saved → data/data_cleaned_auto.csv")

# ── STEP 3: COMPUTE KPIs ──
print("[3/5] Computing KPIs...")
kpis = {
    "Total Revenue (£)":    round(df["TotalPrice"].sum(), 2),
    "Total Orders":         df["InvoiceNo"].nunique(),
    "Unique Customers":     df["CustomerID"].nunique(),
    "Unique Products":      df["Description"].nunique(),
    "Avg Order Value (£)":  round(df["TotalPrice"].sum() / df["InvoiceNo"].nunique(), 2),
    "Top Country":          df.groupby("Country")["TotalPrice"].sum().idxmax(),
}
for k, v in kpis.items():
    print(f"      {k}: {v}")

# ── STEP 4: EXPORT TO EXCEL ──
print("[4/5] Exporting to Excel...")
monthly  = df.groupby(["Year","Month"]).agg(
    Revenue=("TotalPrice","sum"), Orders=("InvoiceNo","nunique"),
    Customers=("CustomerID","nunique")
).reset_index().round(2)

countries = df.groupby("Country").agg(
    Revenue=("TotalPrice","sum"), Customers=("CustomerID","nunique")
).sort_values("Revenue",ascending=False).head(10).reset_index().round(2)

products = df.groupby("Description").agg(
    Revenue=("TotalPrice","sum"), Quantity=("Quantity","sum")
).sort_values("Revenue",ascending=False).head(10).reset_index().round(2)

wb = Workbook()
header_font  = Font(bold=True, color="FFFFFF", size=11)
header_fill  = PatternFill("solid", fgColor="1565C0")
center_align = Alignment(horizontal="center", vertical="center")
thin_border  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

def write_sheet(wb, title, df_data, first=False):
    ws = wb.active if first else wb.create_sheet(title)
    ws.title = title
    ws.append(list(df_data.columns))
    for cell in ws[1]:
        cell.font, cell.fill = header_font, header_fill
        cell.alignment, cell.border = center_align, thin_border
    for row in dataframe_to_rows(df_data, index=False, header=False):
        ws.append(row)
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 20

# KPI sheet
ws_kpi = wb.active
ws_kpi.title = "KPI Summary"
ws_kpi.append(["Metric", "Value"])
for cell in ws_kpi[1]:
    cell.font, cell.fill = header_font, header_fill
    cell.alignment = center_align
for k, v in kpis.items():
    ws_kpi.append([k, v])
for col in ws_kpi.columns:
    ws_kpi.column_dimensions[col[0].column_letter].width = 30

write_sheet(wb, "Monthly Revenue", monthly)
write_sheet(wb, "Top Countries",   countries)
write_sheet(wb, "Top Products",    products)

wb.save("../reports/analytics_report.xlsx")
print("      Saved → reports/analytics_report.xlsx")

# ── STEP 5: SAVE SUMMARY CHARTS ──
print("[5/5] Generating summary charts...")
monthly["Period"] = pd.to_datetime(
    monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(monthly["Period"], monthly["Revenue"],
             color="#1565C0", marker="o", linewidth=2)
axes[0].set_title("Monthly Revenue Trend")
axes[0].set_ylabel("Revenue (£)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].barh(countries["Country"], countries["Revenue"],
             color="#42A5F5")
axes[1].set_title("Top 10 Countries by Revenue")
axes[1].set_xlabel("Revenue (£)")

plt.suptitle("Automated Pipeline — Summary Charts", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/pipeline_summary_charts.png", dpi=150, bbox_inches="tight")
plt.close()
print("      Saved → reports/pipeline_summary_charts.png")

print("\\n" + "=" * 55)
print("   ✅ PIPELINE COMPLETE!")
print("=" * 55)
'''

with open('../scripts/pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_script)

print('✅ Pipeline script saved → scripts/pipeline.py')

### Step 7 — Run the Pipeline Directly from Notebook

In [ ]:
# Run the pipeline directly here in the notebook
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print('=' * 55)
print('   APEXPLANET — AUTOMATED DATA PIPELINE')
print(f'   Run Time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 55)

# ── STEP 1: LOAD ──
print('\n[1/5] Loading raw data...')
df_raw = pd.read_csv('../data/data.csv', encoding='ISO-8859-1')
print(f'      Loaded: {len(df_raw):,} rows')

# ── STEP 2: CLEAN ──
print('[2/5] Cleaning data...')
df_auto = df_raw.copy()
df_auto.drop_duplicates(inplace=True)
df_auto.dropna(subset=['CustomerID'], inplace=True)
df_auto['Description'].fillna('Unknown', inplace=True)
df_auto['InvoiceDate'] = pd.to_datetime(df_auto['InvoiceDate'])
df_auto['CustomerID']  = df_auto['CustomerID'].astype(int).astype(str)
df_auto['Year']  = df_auto['InvoiceDate'].dt.year
df_auto['Month'] = df_auto['InvoiceDate'].dt.month
df_auto['Day']   = df_auto['InvoiceDate'].dt.day
df_auto = df_auto[(df_auto['Quantity'] > 0) & (df_auto['UnitPrice'] > 0)]
for col in ['Quantity', 'UnitPrice']:
    Q1, Q3 = df_auto[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_auto = df_auto[
        (df_auto[col] >= Q1 - 1.5*IQR) &
        (df_auto[col] <= Q3 + 1.5*IQR)
    ]
df_auto['TotalPrice'] = df_auto['Quantity'] * df_auto['UnitPrice']
df_auto.to_csv('../data/data_cleaned_auto.csv', index=False)
print(f'      Clean rows: {len(df_auto):,} | Saved → data/data_cleaned_auto.csv')

# ── STEP 3: KPIs ──
print('[3/5] Computing KPIs...')
kpis = {
    'Total Revenue (£)':   round(df_auto['TotalPrice'].sum(), 2),
    'Total Orders':        df_auto['InvoiceNo'].nunique(),
    'Unique Customers':    df_auto['CustomerID'].nunique(),
    'Unique Products':     df_auto['Description'].nunique(),
    'Avg Order Value (£)': round(df_auto['TotalPrice'].sum() / df_auto['InvoiceNo'].nunique(), 2),
    'Top Country':         df_auto.groupby('Country')['TotalPrice'].sum().idxmax(),
}
for k, v in kpis.items():
    print(f'      {k}: {v}')

# ── STEP 4: EXCEL ──
print('[4/5] Exporting to Excel...')
monthly_auto = df_auto.groupby(['Year','Month']).agg(
    Revenue=('TotalPrice','sum'), Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index().round(2)

countries_auto = df_auto.groupby('Country').agg(
    Revenue=('TotalPrice','sum'), Customers=('CustomerID','nunique')
).sort_values('Revenue', ascending=False).head(10).reset_index().round(2)

products_auto = df_auto.groupby('Description').agg(
    Revenue=('TotalPrice','sum'), Quantity=('Quantity','sum')
).sort_values('Revenue', ascending=False).head(10).reset_index().round(2)

header_font  = Font(bold=True, color='FFFFFF', size=11)
header_fill  = PatternFill('solid', fgColor='1565C0')
center_align = Alignment(horizontal='center', vertical='center')

wb = Workbook()

# KPI Sheet
ws_kpi = wb.active
ws_kpi.title = 'KPI Summary'
ws_kpi.append(['Metric', 'Value'])
for cell in ws_kpi[1]:
    cell.font, cell.fill, cell.alignment = header_font, header_fill, center_align
for k, v in kpis.items():
    ws_kpi.append([k, v])
for col in ws_kpi.columns:
    ws_kpi.column_dimensions[col[0].column_letter].width = 30

# Other Sheets
for sheet_name, data in [
    ('Monthly Revenue', monthly_auto),
    ('Top Countries',   countries_auto),
    ('Top Products',    products_auto)
]:
    ws = wb.create_sheet(sheet_name)
    ws.append(list(data.columns))
    for cell in ws[1]:
        cell.font, cell.fill, cell.alignment = header_font, header_fill, center_align
    for row in dataframe_to_rows(data, index=False, header=False):
        ws.append(row)
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 22

wb.save('../reports/analytics_report.xlsx')
print('      Saved → reports/analytics_report.xlsx')

# ── STEP 5: CHARTS ──
print('[5/5] Generating summary charts...')
monthly_auto['Period'] = pd.to_datetime(
    monthly_auto['Year'].astype(str) + '-' +
    monthly_auto['Month'].astype(str).str.zfill(2)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(monthly_auto['Period'], monthly_auto['Revenue'],
             color='#1565C0', marker='o', linewidth=2)
axes[0].set_title('Monthly Revenue Trend', fontweight='bold')
axes[0].set_ylabel('Revenue (£)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].barh(countries_auto['Country'], countries_auto['Revenue'],
             color='#42A5F5')
axes[1].set_title('Top 10 Countries by Revenue', fontweight='bold')
axes[1].set_xlabel('Revenue (£)')

plt.suptitle('Automated Pipeline — Summary Charts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/pipeline_summary_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('      Saved → reports/pipeline_summary_charts.png')

print('\n' + '=' * 55)
print('   ✅ PIPELINE COMPLETE!')
print('=' * 55)

---
# 📅 DAY 30: GITHUB CLEANUP & REQUIREMENTS

### Step 8 — Generate requirements.txt

In [ ]:
requirements = """pandas>=1.5.0
numpy>=1.23.0
matplotlib>=3.6.0
seaborn>=0.12.0
plotly>=5.11.0
scipy>=1.9.0
scikit-learn>=1.1.0
sqlalchemy>=1.4.0
openpyxl>=3.0.0
fpdf2>=2.5.0
schedule>=1.1.0
notebook>=6.5.0
"""

with open('../requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ requirements.txt saved to project root')
print(requirements)

### Step 9 — Final Project File Checklist

In [ ]:
import os

checklist = {
    'data/data.csv':                           'Raw dataset',
    'data/data_cleaned.csv':                   'Cleaned dataset (Task 1)',
    'data/ecommerce.db':                       'SQLite database (Task 2)',
    'notebooks/task1_eda.ipynb':               'Task 1 notebook',
    'notebooks/task2_sql.ipynb':               'Task 2 notebook',
    'notebooks/task3_visualization.ipynb':     'Task 3 notebook',
    'notebooks/task4_advanced_analytics.ipynb':'Task 4 notebook',
    'notebooks/task5_final.ipynb':             'Task 5 notebook',
    'scripts/task2_queries.sql':               'SQL queries file',
    'scripts/pipeline.py':                     'Automation pipeline script',
    'reports/final_executive_report.pdf':      'Final PDF report',
    'reports/analytics_report.xlsx':           'Excel KPI report',
    'dashboards/executive_dashboard.html':     'Interactive dashboard',
    'requirements.txt':                        'Python dependencies',
    'README.md':                               'Project documentation',
}

print('=' * 65)
print('         📋 FINAL PROJECT FILE CHECKLIST')
print('=' * 65)
all_good = True
for path, desc in checklist.items():
    full_path = f'../{path}'
    exists    = os.path.exists(full_path)
    status    = '✅' if exists else '❌ MISSING'
    if not exists:
        all_good = False
    print(f'  {status}  {path:<45} {desc}')

print('=' * 65)
if all_good:
    print('  🎉 ALL FILES PRESENT — Ready for final submission!')
else:
    print('  ⚠️  Some files are missing — check above and fix before submitting.')
print('=' * 65)

### Step 10 — Git Commands for Final Submission

In [ ]:
print('=== 🚀 FINAL GITHUB PUSH COMMANDS ===')
print()
print('Run these in your terminal from the project folder:')
print()
print('  git add .')
print('  git commit -m "Task 5 complete: Final report, pipeline automation & submission"')
print('  git tag v1.0.0')
print('  git push origin main')
print('  git push origin v1.0.0')
print()
print('=== ✅ SUBMISSION CHECKLIST ===')
print()
steps = [
    'Run all 5 notebooks top to bottom',
    'Push all files to GitHub (git push origin main)',
    'Tag the final commit as v1.0.0',
    'Record screen recording of the project',
    'Upload recording to LinkedIn under Featured',
    'Post LinkedIn update for Task 5',
    'Go to ApexPlanet portal → Manage Task',
    'Verify with offer letter ID and email',
    'Submit LinkedIn link + GitHub link',
]
for i, step in enumerate(steps, 1):
    print(f'  {i}. {step}')

---
## 🎉 INTERNSHIP COMPLETE!

**What we accomplished across all 5 tasks:**

| Task | Topic | Key Output |
|------|-------|------------|
| Task 1 | EDA & Data Cleaning | Cleaned dataset + 7 visualizations + 5 insights |
| Task 2 | SQL for Data Extraction | SQLite DB + 10 business queries + Python integration |
| Task 3 | Visualization & Dashboarding | 13 charts + interactive executive dashboard |
| Task 4 | Advanced Analytics | T-Test + Chi-Square + K-Means clustering + ML models |
| Task 5 | Report, Automation & Presentation | PDF report + Excel export + automated pipeline |

---

### 🏆 Skills Demonstrated
- ✅ Python for Data Analytics (Pandas, NumPy, Matplotlib, Seaborn, Plotly)
- ✅ SQL (SQLite, SQLAlchemy, Window Functions, CTEs)
- ✅ Statistical Analysis (T-Test, Chi-Square, Confidence Intervals)
- ✅ Machine Learning (K-Means Clustering, Linear & Logistic Regression)
- ✅ Dashboard Creation (Interactive Plotly)
- ✅ Report Generation (PDF with fpdf2, Excel with openpyxl)
- ✅ Pipeline Automation
- ✅ GitHub Version Control

---
*ApexPlanet Data Analytics Internship — Task 5 of 5 — COMPLETE! 🏁*  
*ApexPlanet Software Pvt. Ltd. | www.apexplanet.in*